In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import warnings
import os  # 添加这行
import json  # 添加这行
warnings.filterwarnings('ignore')

class ComprehensiveMatAnalysis:
    def __init__(self, mat_file_path, weight_file_path=None, weight_config=None):
        """
        深度分析MAT文件的数据分布和类别不平衡情况
        
        Args:
            mat_file_path: MAT文件路径
            weight_file_path: 外部权重文件路径
            weight_config: 权重配置字典或配置文件路径
        """
        self.mat_file_path = mat_file_path
        self.weight_file_path = weight_file_path
        self.weight_config = weight_config
        self.data = None
        self.labels = None
        self.prob_idx = None
        self.original_data = None  # 仅用于对比分析，不保存到文件
        self.normalization_applied = False  # 标记是否已标准化
        self.external_weights = None  # 从文件读取的权重
        self.weight_metadata = None   # 权重文件的元数据
        self.analysis_results = {}
        
    def load_data(self):
        """加载MAT文件数据"""
        print("🔍 加载MAT文件数据...")
        
        with h5py.File(self.mat_file_path, 'r') as f:
            print(f"📁 MAT文件包含的键: {list(f.keys())}")
            
            # 加载数据
            self.data = np.array(f['data']).transpose()
            self.labels = np.array(f['region']).transpose()  
            self.prob_idx = np.array(f['prob_idx']).transpose()
            
        print(f"✅ 数据加载完成")
        print(f"   - 数据形状: {self.data.shape}")
        print(f"   - 标签形状: {self.labels.shape}")
        print(f"   - prob_idx形状: {self.prob_idx.shape}")
        
        # 新增：自动进行Z-score标准化
        self._apply_zscore_normalization()
        
        # 新增：处理权重配置
        self._process_weight_config()
        
        return self
    
    def _process_weight_config(self):
        """处理权重配置"""
        if self.weight_config:
            if isinstance(self.weight_config, str):
                # 如果是字符串，假设是配置文件路径
                self._load_weight_config_file(self.weight_config)
            elif isinstance(self.weight_config, dict):
                # 如果是字典，直接使用
                if 'effective_number' in self.weight_config:
                    file_path = self.weight_config['effective_number'].get('file_path')
                    if file_path:
                        self.weight_file_path = file_path
        
        # 如果指定了权重文件路径，尝试加载
        if self.weight_file_path:
            try:
                self.load_external_weights()
            except Exception as e:
                print(f"   ⚠️ 加载权重文件失败: {e}")
    
    def _load_weight_config_file(self, config_path):
        """从配置文件加载权重配置"""
        try:
            if config_path.endswith('.yaml') or config_path.endswith('.yml'):
                import yaml
                with open(config_path, 'r', encoding='utf-8') as f:
                    config = yaml.safe_load(f)
                    self.weight_config = config.get('weights', {})
                    if 'effective_number' in self.weight_config:
                        file_path = self.weight_config['effective_number'].get('file_path')
                        if file_path:
                            self.weight_file_path = file_path
            elif config_path.endswith('.json'):
                import json
                with open(config_path, 'r', encoding='utf-8') as f:
                    config = json.load(f)
                    self.weight_config = config.get('weights', {})
                    if 'effective_number' in self.weight_config:
                        file_path = self.weight_config['effective_number'].get('file_path')
                        if file_path:
                            self.weight_file_path = file_path
        except Exception as e:
            print(f"   ⚠️ 加载权重配置文件失败: {e}")
    
    def load_external_weights(self, weight_file_path=None):
        """从外部文件加载预计算的类别权重"""
        if weight_file_path:
            self.weight_file_path = weight_file_path
        
        if not self.weight_file_path:
            print("   ⚠️ 未指定权重文件路径")
            return False
        
        if not os.path.exists(self.weight_file_path):
            print(f"   ⚠️ 权重文件不存在: {self.weight_file_path}")
            return False
        
        try:
            file_ext = os.path.splitext(self.weight_file_path)[1].lower()
            
            if file_ext == '.json':
                with open(self.weight_file_path, 'r', encoding='utf-8') as f:
                    weight_data = json.load(f)
            elif file_ext == '.npy':
                weight_array = np.load(self.weight_file_path)
                weight_data = {
                    'weight_tensor': weight_array.tolist(),
                    'class_weights': {str(i): float(w) for i, w in enumerate(weight_array)}
                }
            elif file_ext == '.npz':
                weight_data_npz = np.load(self.weight_file_path)
                weight_data = {
                    'weight_tensor': weight_data_npz['weights'].tolist(),
                    'metadata': weight_data_npz.get('metadata', {}).item() if 'metadata' in weight_data_npz else {}
                }
            elif file_ext == '.csv':
                import pandas as pd
                df = pd.read_csv(self.weight_file_path)
                if 'class_id' in df.columns and 'weight' in df.columns:
                    weight_data = {
                        'class_weights': dict(zip(df['class_id'].astype(str), df['weight'])),
                        'weight_tensor': df['weight'].tolist()
                    }
                else:
                    raise ValueError("CSV文件必须包含'class_id'和'weight'列")
            else:
                raise ValueError(f"不支持的文件格式: {file_ext}")
            
            # 存储权重数据
            self.external_weights = weight_data
            self.weight_metadata = weight_data.get('metadata', {})
            
            print(f"   ✅ 成功加载外部权重文件: {self.weight_file_path}")
            
            # 验证权重兼容性
            validation_result = self.validate_weights_compatibility()
            if not all(validation_result.values()):
                print(f"   ⚠️ 权重兼容性检查发现问题: {validation_result}")
            
            return True
            
        except Exception as e:
            print(f"   ❌ 加载权重文件失败: {e}")
            return False
    
    def validate_weights_compatibility(self):
        """验证权重文件与当前数据的兼容性"""
        if not self.external_weights:
            return {'error': '没有外部权重数据'}
        
        checks = {
            'class_count_match': False,
            'weight_format_valid': False,
            'metadata_consistent': False,
            'weight_values_reasonable': False
        }
        
        try:
            # 检查是否已有基础分析结果
            if 'basic_stats' not in self.analysis_results:
                print("   ℹ️ 需要先运行基础统计分析才能完全验证权重兼容性")
                return checks
            
            expected_classes = self.analysis_results['basic_stats']['num_classes']
            
            # 检查类别数量是否匹配
            if 'class_weights' in self.external_weights:
                actual_classes = len(self.external_weights['class_weights'])
                checks['class_count_match'] = (actual_classes == expected_classes)
            elif 'weight_tensor' in self.external_weights:
                actual_classes = len(self.external_weights['weight_tensor'])
                checks['class_count_match'] = (actual_classes == expected_classes)
            
            # 检查权重格式是否正确
            if 'weight_tensor' in self.external_weights:
                weights = self.external_weights['weight_tensor']
                checks['weight_format_valid'] = (
                    isinstance(weights, list) and 
                    len(weights) > 0 and 
                    all(isinstance(w, (int, float)) for w in weights)
                )
            
            # 检查权重值是否合理
            if checks['weight_format_valid']:
                weights = self.external_weights['weight_tensor']
                checks['weight_values_reasonable'] = (
                    all(w > 0 for w in weights) and  # 所有权重为正
                    min(weights) > 0 and  # 最小权重大于0
                    max(weights) / min(weights) < 1000  # 权重比例不会过于极端
                )
            
            # 检查元数据一致性
            if self.weight_metadata:
                metadata_expected_classes = self.weight_metadata.get('num_classes')
                if metadata_expected_classes:
                    checks['metadata_consistent'] = (metadata_expected_classes == expected_classes)
                else:
                    checks['metadata_consistent'] = True  # 没有元数据也算通过
            else:
                checks['metadata_consistent'] = True
            
        except Exception as e:
            print(f"   ⚠️ 验证权重兼容性时出错: {e}")
        
        return checks
    
    def merge_weights_with_analysis(self):
        """将外部权重集成到分析结果中"""
        if not self.external_weights or 'weight_strategies' not in self.analysis_results:
            return
        
        try:
            # 将外部权重添加到权重策略中
            if 'class_weights' in self.external_weights:
                # 转换字符串键为整数键
                external_dict = {}
                for k, v in self.external_weights['class_weights'].items():
                    external_dict[int(k)] = float(v)
                
                # 添加到权重策略
                method_name = self.weight_metadata.get('method', 'external')
                strategy_name = f"{method_name}_external"
                self.analysis_results['weight_strategies'][strategy_name] = external_dict
                
                print(f"   ✅ 外部权重已集成到分析结果中: {strategy_name}")
            
        except Exception as e:
            print(f"   ⚠️ 集成外部权重失败: {e}")
    
    def export_weights_to_file(self, output_path, weight_strategy='effective_number', **kwargs):
        """将当前分析的权重导出到文件"""
        if 'weight_strategies' not in self.analysis_results:
            print("   ⚠️ 请先运行calculate_class_weights()方法")
            return False
        
        weights = self.analysis_results['weight_strategies']
        if weight_strategy not in weights:
            print(f"   ⚠️ 权重策略 '{weight_strategy}' 不存在")
            return False
        
        try:
            # 准备导出数据
            strategy_weights = weights[weight_strategy]
            
            # 构建权重数组（确保按类别ID顺序）
            num_classes = self.analysis_results['basic_stats']['num_classes']
            weight_tensor = []
            for i in range(num_classes):
                weight_tensor.append(strategy_weights.get(i, 1.0))
            
            # 准备元数据
            metadata = {
                'method': weight_strategy,
                'dataset': os.path.basename(self.mat_file_path),
                'num_classes': num_classes,
                'creation_time': pd.Timestamp.now().isoformat(),
                'normalization_applied': self.normalization_applied
            }
            
            # 添加策略特定的参数
            if 'beta' in kwargs:
                metadata['beta'] = kwargs['beta']
            
            if 'class_distribution' in self.analysis_results:
                metadata['imbalance_ratio'] = self.analysis_results['class_distribution']['imbalance_ratio']
            
            # 构建完整数据
            export_data = {
                'metadata': metadata,
                'class_weights': {str(k): v for k, v in strategy_weights.items()},
                'weight_tensor': weight_tensor
            }
            
            # 根据文件扩展名选择导出格式
            file_ext = os.path.splitext(output_path)[1].lower()
            
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            
            if file_ext == '.json':
                with open(output_path, 'w', encoding='utf-8') as f:
                    json.dump(export_data, f, indent=2, ensure_ascii=False)
            elif file_ext == '.npy':
                np.save(output_path, np.array(weight_tensor))
            elif file_ext == '.npz':
                np.savez(output_path, weights=np.array(weight_tensor), metadata=metadata)
            elif file_ext == '.csv':
                import pandas as pd
                df = pd.DataFrame({
                    'class_id': list(range(num_classes)),
                    'weight': weight_tensor
                })
                df.to_csv(output_path, index=False)
            else:
                # 默认使用JSON格式
                with open(output_path + '.json', 'w', encoding='utf-8') as f:
                    json.dump(export_data, f, indent=2, ensure_ascii=False)
                output_path += '.json'
            
            print(f"   ✅ 权重已导出到: {output_path}")
            return True
            
        except Exception as e:
            print(f"   ❌ 导出权重失败: {e}")
            return False
    
    def _apply_zscore_normalization(self):
        """应用Z-score标准化到所有特征维度"""
        print("\n🔧 应用Z-score标准化...")
        
        # 保存原始数据的引用（仅用于对比分析）
        self.original_data = self.data.copy()
        
        # 计算每个特征的均值和标准差
        feature_means = np.mean(self.data, axis=0)
        feature_stds = np.std(self.data, axis=0)
        
        # 识别零方差特征
        zero_variance_mask = feature_stds == 0
        zero_variance_count = np.sum(zero_variance_mask)
        
        if zero_variance_count > 0:
            print(f"   ⚠️ 发现 {zero_variance_count} 个零方差特征，将保持原值")
        
        # 应用Z-score标准化
        # 对于零方差特征，保持原值不变
        normalized_data = np.zeros_like(self.data)
        
        for i in range(self.data.shape[1]):
            if feature_stds[i] != 0:
                normalized_data[:, i] = (self.data[:, i] - feature_means[i]) / feature_stds[i]
            else:
                normalized_data[:, i] = self.data[:, i]
        
        # 更新数据为标准化后的数据
        self.data = normalized_data
        self.normalization_applied = True
        
        # 打印标准化摘要信息
        print(f"   ✅ Z-score标准化完成")
        print(f"   📊 处理特征数: {self.data.shape[1]}")
        print(f"   📊 零方差特征数: {zero_variance_count}")
        print(f"   📊 标准化后数据范围: [{np.min(self.data):.3f}, {np.max(self.data):.3f}]")
        print(f"   📊 标准化后均值: {np.mean(self.data):.6f}")
        print(f"   📊 标准化后标准差: {np.std(self.data):.6f}")
    
    def analyze_basic_statistics(self):
        """基础统计分析"""
        print("\n📊 基础统计分析...")
        
        results = {}
        
        # 数据基本信息
        results['data_info'] = {
            'total_samples': len(self.data),
            'total_features': self.data.shape[1],
            'data_type': str(self.data.dtype),
            'memory_usage_mb': self.data.nbytes / (1024**2)
        }
        
        # 标签信息分析
        if self.labels.ndim > 1 and self.labels.shape[1] > 1:
            # One-hot编码格式
            label_indices = np.argmax(self.labels, axis=1)
            results['label_format'] = 'one_hot'
            results['num_classes'] = self.labels.shape[1]
        else:
            # 直接标签格式
            label_indices = self.labels.flatten()
            results['label_format'] = 'direct'
            results['num_classes'] = len(np.unique(label_indices))
        
        results['label_indices'] = label_indices
        results['unique_labels'] = np.unique(label_indices)
        results['label_range'] = (int(np.min(label_indices)), int(np.max(label_indices)))
        
        # prob_idx分析
        unique_prob_idx = np.unique(self.prob_idx.flatten())
        results['prob_idx_info'] = {
            'unique_values': unique_prob_idx.tolist(),
            'counts': [int(np.sum(self.prob_idx == idx)) for idx in unique_prob_idx]
        }
        
        # 数据质量检查
        results['data_quality'] = {
            'has_nan': bool(np.isnan(self.data).any()),
            'has_inf': bool(np.isinf(self.data).any()),
            'nan_count': int(np.isnan(self.data).sum()),
            'inf_count': int(np.isinf(self.data).sum()),
            'zero_samples': int(np.sum(np.all(self.data == 0, axis=1))),
            'data_range': (float(np.min(self.data)), float(np.max(self.data))),
            'data_mean': float(np.mean(self.data)),
            'data_std': float(np.std(self.data))
        }
        
        self.analysis_results['basic_stats'] = results
        
        # 打印关键信息
        print(f"   📋 样本数量: {results['data_info']['total_samples']:,}")
        print(f"   📋 特征数量: {results['data_info']['total_features']:,}")
        print(f"   📋 类别数量: {results['num_classes']}")
        print(f"   📋 标签范围: {results['label_range']}")
        print(f"   📋 标签格式: {results['label_format']}")
        print(f"   📋 prob_idx取值: {results['prob_idx_info']['unique_values']}")
        print(f"   📋 数据范围: [{results['data_quality']['data_range'][0]:.3f}, {results['data_quality']['data_range'][1]:.3f}]")
        
        if results['data_quality']['has_nan'] or results['data_quality']['has_inf']:
            print(f"   ⚠️ 数据质量问题: NaN={results['data_quality']['nan_count']}, Inf={results['data_quality']['inf_count']}")
        
        # 新增：标准化信息报告
        if self.normalization_applied:
            print(f"   ✅ 已应用Z-score标准化 (341个特征维度)")
            print(f"   📊 标准化后数据范围: [{np.min(self.data):.3f}, {np.max(self.data):.3f}]")
        
        return self
    
    def analyze_class_distribution(self, detailed=True):
        """类别分布分析"""
        print("\n📈 类别分布分析...")
        
        label_indices = self.analysis_results['basic_stats']['label_indices']
        
        # 计算类别分布
        class_counts = Counter(label_indices)
        total_samples = len(label_indices)
        
        # 基础分布统计
        distribution_stats = {
            'class_counts': dict(class_counts),
            'total_samples': total_samples,
            'num_classes_present': len(class_counts),
            'expected_num_classes': self.analysis_results['basic_stats']['num_classes']
        }
        
        # 计算类别统计
        counts_array = np.array(list(class_counts.values()))
        distribution_stats.update({
            'mean_samples_per_class': float(np.mean(counts_array)),
            'std_samples_per_class': float(np.std(counts_array)),
            'min_samples': int(np.min(counts_array)),
            'max_samples': int(np.max(counts_array)),
            'median_samples': float(np.median(counts_array)),
            'imbalance_ratio': float(np.max(counts_array) / np.min(counts_array))
        })
        
        # 识别缺失的类别
        all_possible_classes = set(range(self.analysis_results['basic_stats']['num_classes']))
        present_classes = set(class_counts.keys())
        missing_classes = sorted(list(all_possible_classes - present_classes))
        distribution_stats['missing_classes'] = missing_classes
        distribution_stats['missing_classes_count'] = len(missing_classes)
        
        # 类别分组分析
        percentiles = [10, 25, 50, 75, 90]
        percentile_values = np.percentile(counts_array, percentiles)
        distribution_stats['sample_count_percentiles'] = dict(zip(percentiles, percentile_values))
        
        # 不平衡程度分类
        imbalance_ratio = distribution_stats['imbalance_ratio']
        if imbalance_ratio <= 2:
            imbalance_level = "轻微不平衡"
        elif imbalance_ratio <= 5:
            imbalance_level = "中等不平衡"
        elif imbalance_ratio <= 10:
            imbalance_level = "严重不平衡"
        else:
            imbalance_level = "极度不平衡"
        
        distribution_stats['imbalance_level'] = imbalance_level
        
        # 分析prob_idx对类别分布的影响
        prob_idx_flat = self.prob_idx.flatten()
        prob_idx_analysis = {}
        
        for prob_val in np.unique(prob_idx_flat):
            mask = prob_idx_flat == prob_val
            subset_labels = label_indices[mask]
            subset_counts = Counter(subset_labels)
            
            prob_idx_analysis[int(prob_val)] = {
                'sample_count': int(np.sum(mask)),
                'unique_classes': len(subset_counts),
                'class_distribution': dict(subset_counts),
                'most_common_class': int(subset_counts.most_common(1)[0][0]) if subset_counts else None,
                'most_common_count': int(subset_counts.most_common(1)[0][1]) if subset_counts else 0
            }
        
        distribution_stats['prob_idx_analysis'] = prob_idx_analysis
        
        self.analysis_results['class_distribution'] = distribution_stats
        
        # 打印关键统计
        print(f"   📊 总样本数: {total_samples:,}")
        print(f"   📊 实际类别数: {distribution_stats['num_classes_present']}/{distribution_stats['expected_num_classes']}")
        print(f"   📊 缺失类别数: {distribution_stats['missing_classes_count']}")
        print(f"   📊 不平衡比例: {imbalance_ratio:.2f} ({imbalance_level})")
        print(f"   📊 样本数范围: {distribution_stats['min_samples']} - {distribution_stats['max_samples']}")
        print(f"   📊 平均每类样本: {distribution_stats['mean_samples_per_class']:.1f} ± {distribution_stats['std_samples_per_class']:.1f}")
        
        if missing_classes:
            print(f"   ⚠️ 缺失的类别: {missing_classes[:10]}{'...' if len(missing_classes) > 10 else ''}")
        
        return self
    
    def calculate_class_weights(self):
        """计算多种类别权重策略 - 增强版"""
        print("\n⚖️ 计算类别权重...")
        
        label_indices = self.analysis_results['basic_stats']['label_indices']
        class_counts = self.analysis_results['class_distribution']['class_counts']
        num_classes = self.analysis_results['basic_stats']['num_classes']
        total_samples = len(label_indices)
        
        weight_strategies = {}
        
        # 1. 平衡权重 (balanced) - 保持原有代码
        try:
            present_classes = np.array(list(class_counts.keys()))
            balanced_weights = compute_class_weight(
                'balanced', 
                classes=present_classes, 
                y=label_indices
            )
            balanced_weight_dict = dict(zip(present_classes, balanced_weights))
            
            missing_classes = self.analysis_results['class_distribution']['missing_classes']
            max_balanced_weight = max(balanced_weights) if len(balanced_weights) > 0 else 1.0
            for missing_class in missing_classes:
                balanced_weight_dict[missing_class] = max_balanced_weight * 2
            
            weight_strategies['balanced'] = balanced_weight_dict
        except Exception as e:
            print(f"   ⚠️ 平衡权重计算失败: {e}")
            weight_strategies['balanced'] = {}
        
        # 2. 反频率权重 (inverse frequency)
        inverse_freq_weights = {}
        for class_id in range(num_classes):
            count = class_counts.get(class_id, 0)
            if count > 0:
                inverse_freq_weights[class_id] = total_samples / (num_classes * count)
            else:
                # 为缺失类别分配最大权重
                max_count = max(class_counts.values()) if class_counts else 1
                inverse_freq_weights[class_id] = total_samples / (num_classes * 1)  # 假设至少有1个样本
        
        weight_strategies['inverse_frequency'] = inverse_freq_weights
        
        # 3. 平方根权重 (sqrt balanced)
        sqrt_weights = {}
        total_sqrt_samples = np.sqrt(total_samples)
        for class_id in range(num_classes):
            count = class_counts.get(class_id, 0)
            if count > 0:
                sqrt_weights[class_id] = total_sqrt_samples / np.sqrt(count)
            else:
                sqrt_weights[class_id] = total_sqrt_samples  # 给缺失类别高权重
        
        weight_strategies['sqrt_balanced'] = sqrt_weights
        
        # 4. 对数权重 (log balanced)
        log_weights = {}
        for class_id in range(num_classes):
            count = class_counts.get(class_id, 0)
            if count > 0:
                log_weights[class_id] = np.log(total_samples / count)
            else:
                max_count = max(class_counts.values()) if class_counts else 1
                log_weights[class_id] = np.log(total_samples)  # 给缺失类别高权重
        
        weight_strategies['log_balanced'] = log_weights
        
        # 5. 有效样本数权重 (effective number based)
        def compute_effective_number_weights(class_counts_dict, beta='auto', num_classes=None, return_type='dict'):
            """
            计算增强版Effective Number权重（基于Cui et al., 2019）
            
            Args:
                class_counts_dict: 类别样本数字典
                beta: 超参数，'auto'表示自动选择，或具体数值
                num_classes: 总类别数
                return_type: 返回类型 ('dict', 'numpy', 'torch')
            
            Returns:
                tuple: (effective_weights, used_beta, metadata)
            """
            import numpy as np
            
            if num_classes is None:
                num_classes = len(class_counts_dict)
            
            # 计算不平衡比例用于自动选择beta
            present_counts = [v for v in class_counts_dict.values() if v > 0]
            if present_counts:
                imbalance_ratio = max(present_counts) / min(present_counts)
            else:
                imbalance_ratio = 1.0
            
            # 自动选择beta值（根据论文观察）
            if beta == 'auto':
                if num_classes > 50:  # 细粒度数据集（如CIFAR-100, iNaturalist）
                    if imbalance_ratio < 50:
                        beta = 0.99
                    elif imbalance_ratio < 100:
                        beta = 0.999
                    else:
                        beta = 0.9999
                else:  # 粗粒度数据集（如CIFAR-10）
                    if imbalance_ratio < 100:
                        beta = 0.999
                    else:
                        beta = 0.9999
                
                print(f"   🎯 自动选择beta={beta} (类别数={num_classes}, 不平衡比例={imbalance_ratio:.2f})")
            
            # 确保beta在有效范围内
            beta = max(0.0, min(0.9999, float(beta)))
            
            effective_weights = {}
            
            # 计算每个类别的effective number和权重
            for class_id in range(num_classes):
                count = class_counts_dict.get(class_id, 0)
                if count > 0:
                    # 论文公式: En = (1 - β^n) / (1 - β)
                    if beta == 0:
                        effective_num = 1  # 特殊情况：无重叠
                    else:
                        effective_num = (1 - beta**count) / (1 - beta)
                    weight = 1 / effective_num
                else:
                    # 零样本类别分配高权重
                    if beta == 0:
                        weight = 1.0
                    else:
                        weight = 1 / (1 - beta)  # 最大可能的effective number权重
                
                effective_weights[class_id] = weight
            
            # 标准化权重（使均值为1，总和为类别数）
            weight_values = list(effective_weights.values())
            if weight_values:
                mean_weight = np.mean(weight_values)
                for class_id in effective_weights:
                    effective_weights[class_id] = effective_weights[class_id] / mean_weight
            
            # 准备元数据
            metadata = {
                'beta': beta,
                'imbalance_ratio': imbalance_ratio,
                'num_classes': num_classes,
                'present_classes': len(present_counts),
                'weight_range': (min(effective_weights.values()), max(effective_weights.values())),
                'method': 'effective_number_cui2019'
            }
            
            # 根据返回类型转换
            if return_type == 'numpy':
                weight_array = np.array([effective_weights.get(i, 1.0) for i in range(num_classes)])
                return weight_array, beta, metadata
            elif return_type == 'torch':
                try:
                    import torch
                    weight_tensor = torch.FloatTensor([effective_weights.get(i, 1.0) for i in range(num_classes)])
                    return weight_tensor, beta, metadata
                except ImportError:
                    print("   ⚠️ PyTorch未安装，返回numpy数组")
                    weight_array = np.array([effective_weights.get(i, 1.0) for i in range(num_classes)])
                    return weight_array, beta, metadata
            else:  # 'dict'
                return effective_weights, beta, metadata

        # 5. 增强版有效样本数权重 (effective number based)
        effective_weights, used_beta, en_metadata = compute_effective_number_weights(
            class_counts, beta='auto', num_classes=num_classes, return_type='dict'
        )
        weight_strategies['effective_number'] = effective_weights
        
        # 记录使用的beta值和元数据
        self.analysis_results['effective_number_beta'] = used_beta
        self.analysis_results['effective_number_metadata'] = en_metadata
        
        # 6. 多个beta值的effective number权重（供对比）
        beta_variants = [0.9, 0.99, 0.999, 0.9999]
        for beta_val in beta_variants:
            if beta_val != used_beta:  # 避免重复
                en_weights_variant, _, _ = compute_effective_number_weights(
                    class_counts, beta=beta_val, num_classes=num_classes, return_type='dict'
                )
                weight_strategies[f'effective_number_beta_{beta_val}'] = en_weights_variant
        


        
        # 6. 分组权重 (prob_idx based)
        prob_idx_weights = {}
        prob_idx_analysis = self.analysis_results['class_distribution']['prob_idx_analysis']
        
        # 为每个prob_idx组计算独立的权重
        for class_id in range(num_classes):
            weights_by_prob = []
            total_count = 0
            
            for prob_val, prob_info in prob_idx_analysis.items():
                class_count_in_prob = prob_info['class_distribution'].get(class_id, 0)
                prob_total = prob_info['sample_count']
                
                if class_count_in_prob > 0 and prob_total > 0:
                    # 在该prob_idx组中的权重
                    local_weight = prob_total / (len(prob_info['class_distribution']) * class_count_in_prob)
                    weights_by_prob.append(local_weight)
                    total_count += class_count_in_prob
            
            if weights_by_prob:
                prob_idx_weights[class_id] = np.mean(weights_by_prob)
            else:
                prob_idx_weights[class_id] = 10.0  # 给缺失类别高权重
        
        weight_strategies['prob_idx_based'] = prob_idx_weights
        
        # 标准化所有权重策略
        for strategy_name, weights in weight_strategies.items():
            if weights:
                weight_array = np.array(list(weights.values()))
                if np.any(weight_array > 0):  # 确保有有效权重
                    normalized_weights = weight_array / np.mean(weight_array)
                    weight_strategies[strategy_name] = dict(zip(weights.keys(), normalized_weights))
        
        self.analysis_results['weight_strategies'] = weight_strategies
        
        # 新增：如果有外部权重文件，加载并集成
        if self.external_weights:
            self.merge_weights_with_analysis()
            print(f"   ✅ 已集成外部权重文件")
        
        # 增强的权重统计报告
        print(f"   ✅ 生成了 {len(weight_strategies)} 种权重策略")
        print(f"   🎯 推荐的Effective Number beta值: {used_beta}")
        print(f"   📊 不平衡比例: {en_metadata['imbalance_ratio']:.2f}")
        
        for strategy_name, weights in weight_strategies.items():
            if weights:
                weight_values = list(weights.values())
                if strategy_name == 'effective_number':
                    marker = "👉 "
                else:
                    marker = "   "
                print(f"{marker}{strategy_name}: 范围[{min(weight_values):.3f}, {max(weight_values):.3f}], "
                    f"均值={np.mean(weight_values):.3f}")
        
        return self
    
    def analyze_feature_distribution(self):
        """特征分布分析"""
        print("\n🔍 特征分布分析...")
        
        feature_stats = {}
        
        # 基础特征统计
        feature_stats['means'] = np.mean(self.data, axis=0)
        feature_stats['stds'] = np.std(self.data, axis=0)
        feature_stats['mins'] = np.min(self.data, axis=0)
        feature_stats['maxs'] = np.max(self.data, axis=0)
        feature_stats['medians'] = np.median(self.data, axis=0)
        
        # 特征质量检查
        feature_stats['zero_variance_features'] = np.where(feature_stats['stds'] == 0)[0].tolist()
        feature_stats['low_variance_features'] = np.where(feature_stats['stds'] < 0.01)[0].tolist()
        feature_stats['high_variance_features'] = np.where(feature_stats['stds'] > np.percentile(feature_stats['stds'], 95))[0].tolist()
        
        # 特征相关性（采样分析以节省时间）
        if self.data.shape[1] > 1000:
            # 随机采样一部分特征进行相关性分析
            sample_features = np.random.choice(self.data.shape[1], 1000, replace=False)
            sample_data = self.data[:, sample_features]
        else:
            sample_data = self.data
            sample_features = np.arange(self.data.shape[1])
        
        correlation_matrix = np.corrcoef(sample_data.T)
        high_corr_pairs = []
        
        for i in range(len(sample_features)):
            for j in range(i+1, len(sample_features)):
                if abs(correlation_matrix[i, j]) > 0.9:
                    high_corr_pairs.append((int(sample_features[i]), int(sample_features[j]), float(correlation_matrix[i, j])))
        
        feature_stats['high_correlation_pairs'] = high_corr_pairs[:50]  # 只保留前50个
        
        # 异常值检测
        q25 = np.percentile(self.data, 25, axis=0)
        q75 = np.percentile(self.data, 75, axis=0)
        iqr = q75 - q25
        
        outlier_bounds_lower = q25 - 1.5 * iqr
        outlier_bounds_upper = q75 + 1.5 * iqr
        
        outliers_per_feature = []
        for i in range(self.data.shape[1]):
            outliers = np.sum((self.data[:, i] < outlier_bounds_lower[i]) | (self.data[:, i] > outlier_bounds_upper[i]))
            outliers_per_feature.append(int(outliers))
        
        feature_stats['outliers_per_feature'] = outliers_per_feature
        feature_stats['features_with_many_outliers'] = np.where(np.array(outliers_per_feature) > len(self.data) * 0.05)[0].tolist()
        
        self.analysis_results['feature_analysis'] = feature_stats
        
        print(f"   📊 零方差特征: {len(feature_stats['zero_variance_features'])}")
        print(f"   📊 低方差特征: {len(feature_stats['low_variance_features'])}")
        print(f"   📊 高相关特征对: {len(feature_stats['high_correlation_pairs'])}")
        print(f"   📊 异常值较多的特征: {len(feature_stats['features_with_many_outliers'])}")
        
        # 新增：标准化效果统计
        if self.normalization_applied and self.original_data is not None:
            self._analyze_normalization_effect(feature_stats)
        
        return self
    
    def _analyze_normalization_effect(self, feature_stats):
        """分析标准化前后的效果对比"""
        print("   🔍 分析标准化效果...")
        
        # 计算原始数据的特征统计
        original_means = np.mean(self.original_data, axis=0)
        original_stds = np.std(self.original_data, axis=0)
        original_zero_variance = np.sum(original_stds == 0)
        
        # 计算标准化效果
        normalization_effect = {
            'original_feature_mean_range': (float(np.min(original_means)), float(np.max(original_means))),
            'original_feature_std_range': (float(np.min(original_stds[original_stds > 0])) if np.any(original_stds > 0) else 0, 
                                         float(np.max(original_stds))),
            'normalized_feature_mean_range': (float(np.min(feature_stats['means'])), float(np.max(feature_stats['means']))),
            'normalized_feature_std_range': (float(np.min(feature_stats['stds'][feature_stats['stds'] > 0])) if np.any(feature_stats['stds'] > 0) else 0,
                                           float(np.max(feature_stats['stds']))),
            'zero_variance_before': int(original_zero_variance),
            'zero_variance_after': len(feature_stats['zero_variance_features']),
            'mean_std_before_normalization': float(np.mean(original_stds)),
            'mean_std_after_normalization': float(np.mean(feature_stats['stds']))
        }
        
        feature_stats['normalization_effect'] = normalization_effect
        
        print(f"   📊 标准化前特征均值范围: [{normalization_effect['original_feature_mean_range'][0]:.3f}, {normalization_effect['original_feature_mean_range'][1]:.3f}]")
        print(f"   📊 标准化后特征均值范围: [{normalization_effect['normalized_feature_mean_range'][0]:.6f}, {normalization_effect['normalized_feature_mean_range'][1]:.6f}]")
        print(f"   📊 标准化前特征标准差均值: {normalization_effect['mean_std_before_normalization']:.3f}")
        print(f"   📊 标准化后特征标准差均值: {normalization_effect['mean_std_after_normalization']:.6f}")
    
    def generate_comprehensive_report(self, save_path="mat_analysis_report.txt"):
        """生成综合分析报告"""
        print(f"\n📝 生成综合分析报告: {save_path}")
        
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write("MAT文件深度数据分析报告\n")
            f.write("=" * 60 + "\n\n")
            f.write(f"文件路径: {self.mat_file_path}\n")
            f.write(f"分析时间: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            
            # 基础统计
            basic_stats = self.analysis_results['basic_stats']
            f.write("1. 基础数据统计\n")
            f.write("-" * 30 + "\n")
            f.write(f"总样本数: {basic_stats['data_info']['total_samples']:,}\n")
            f.write(f"特征维度: {basic_stats['data_info']['total_features']:,}\n")
            f.write(f"类别数量: {basic_stats['num_classes']}\n")
            f.write(f"标签格式: {basic_stats['label_format']}\n")
            f.write(f"标签范围: {basic_stats['label_range']}\n")
            f.write(f"内存占用: {basic_stats['data_info']['memory_usage_mb']:.2f} MB\n\n")
            
            # 新增：1.5. 数据标准化信息
            if self.normalization_applied:
                f.write("1.5. 数据预处理\n")
                f.write("-" * 30 + "\n")
                f.write("标准化方法: Z-score标准化\n")
                f.write("标准化范围: 所有341个特征维度\n")
                f.write("处理方式: 每个特征独立计算均值和标准差\n")
                f.write("零方差特征处理: 保持原值不变\n")
                
                # 添加标准化效果统计
                if 'feature_analysis' in self.analysis_results and 'normalization_effect' in self.analysis_results['feature_analysis']:
                    norm_effect = self.analysis_results['feature_analysis']['normalization_effect']
                    f.write(f"标准化前特征均值范围: [{norm_effect['original_feature_mean_range'][0]:.3f}, {norm_effect['original_feature_mean_range'][1]:.3f}]\n")
                    f.write(f"标准化后特征均值范围: [{norm_effect['normalized_feature_mean_range'][0]:.6f}, {norm_effect['normalized_feature_mean_range'][1]:.6f}]\n")
                    f.write(f"标准化前零方差特征: {norm_effect['zero_variance_before']}\n")
                    f.write(f"标准化后零方差特征: {norm_effect['zero_variance_after']}\n")
                f.write("\n")
            
            # 数据质量
            quality = basic_stats['data_quality']
            f.write("2. 数据质量评估\n")
            f.write("-" * 30 + "\n")
            f.write(f"数据范围: [{quality['data_range'][0]:.6f}, {quality['data_range'][1]:.6f}]\n")
            f.write(f"数据均值: {quality['data_mean']:.6f}\n")
            f.write(f"数据标准差: {quality['data_std']:.6f}\n")
            f.write(f"NaN值数量: {quality['nan_count']}\n")
            f.write(f"无穷值数量: {quality['inf_count']}\n")
            f.write(f"全零样本数: {quality['zero_samples']}\n\n")
            
            # 类别分布分析
            class_dist = self.analysis_results['class_distribution']
            f.write("3. 类别分布分析\n")
            f.write("-" * 30 + "\n")
            f.write(f"实际类别数: {class_dist['num_classes_present']}/{class_dist['expected_num_classes']}\n")
            f.write(f"缺失类别数: {class_dist['missing_classes_count']}\n")
            f.write(f"不平衡比例: {class_dist['imbalance_ratio']:.2f} ({class_dist['imbalance_level']})\n")
            f.write(f"样本数统计:\n")
            f.write(f"  最少: {class_dist['min_samples']}\n")
            f.write(f"  最多: {class_dist['max_samples']}\n")
            f.write(f"  平均: {class_dist['mean_samples_per_class']:.1f}\n")
            f.write(f"  中位数: {class_dist['median_samples']:.1f}\n")
            f.write(f"  标准差: {class_dist['std_samples_per_class']:.1f}\n\n")
            
            # 权重推荐
            if 'weight_strategies' in self.analysis_results:
                weights = self.analysis_results['weight_strategies']
                f.write("4. 权重策略推荐\n")
                f.write("-" * 30 + "\n")
                
                # 检查是否有外部权重
                external_weight_used = False
                external_strategy_name = None
                for strategy_name in weights.keys():
                    if strategy_name.endswith('_external'):
                        external_weight_used = True
                        external_strategy_name = strategy_name
                        break
                

                f.write("-" * 30 + "\n")
                
                # 根据不平衡程度推荐策略
                imbalance_ratio = class_dist['imbalance_ratio']
                if imbalance_ratio <= 2:
                    recommended = "balanced"
                    f.write("推荐策略: 平衡权重 (数据不平衡程度较轻)\n")
                elif imbalance_ratio <= 5:
                    recommended = "sqrt_balanced"
                    f.write("推荐策略: 平方根平衡权重 (中等不平衡)\n")
                elif imbalance_ratio <= 10:
                    recommended = "effective_number"
                    f.write("推荐策略: 有效样本数权重 (严重不平衡)\n")
                else:
                    recommended = "inverse_frequency"
                    f.write("推荐策略: 反频率权重 (极度不平衡)\n")
                
                f.write(f"\n各策略权重范围:\n")
                for strategy_name, strategy_weights in weights.items():
                    if strategy_weights:
                        values = list(strategy_weights.values())
                        marker = "👉 " if strategy_name == recommended else "   "
                        f.write(f"{marker}{strategy_name}: [{min(values):.3f}, {max(values):.3f}], 均值={np.mean(values):.3f}\n")
                
                f.write(f"\n推荐权重详情 ({recommended}):\n")
                if recommended in weights and weights[recommended]:
                    sorted_weights = sorted(weights[recommended].items(), key=lambda x: x[1], reverse=True)
                    
                    # 显示前20个最高权重的类别
                    f.write("最高权重类别 (前20):\n")
                    for i, (class_id, weight) in enumerate(sorted_weights[:20]):
                        count = class_dist['class_counts'].get(class_id, 0)
                        f.write(f"  类别{class_id}: 权重={weight:.4f}, 样本数={count}\n")
                    
                    # 显示最低权重的类别
                    f.write("\n最低权重类别 (后10):\n")
                    for i, (class_id, weight) in enumerate(sorted_weights[-10:]):
                        count = class_dist['class_counts'].get(class_id, 0)
                        f.write(f"  类别{class_id}: 权重={weight:.4f}, 样本数={count}\n")
                f.write("\n")
            
            # prob_idx分析
            f.write("5. prob_idx分组分析\n")
            f.write("-" * 30 + "\n")
            prob_analysis = class_dist['prob_idx_analysis']
            for prob_val, info in prob_analysis.items():
                f.write(f"prob_idx = {prob_val}:\n")
                f.write(f"  样本数: {info['sample_count']}\n")
                f.write(f"  类别数: {info['unique_classes']}\n")
                f.write(f"  最多类别: {info['most_common_class']} ({info['most_common_count']}样本)\n")
                
                # 显示该组中样本数最多的前5个类别
                sorted_classes = sorted(info['class_distribution'].items(), key=lambda x: x[1], reverse=True)[:5]
                f.write(f"  主要类别: {', '.join([f'{c}({n})' for c, n in sorted_classes])}\n\n")
            
            # 特征分析
            if 'feature_analysis' in self.analysis_results:
                feature_analysis = self.analysis_results['feature_analysis']
                f.write("6. 特征分析\n")
                f.write("-" * 30 + "\n")
                f.write(f"零方差特征数: {len(feature_analysis['zero_variance_features'])}\n")
                f.write(f"低方差特征数: {len(feature_analysis['low_variance_features'])}\n")
                f.write(f"高相关特征对数: {len(feature_analysis['high_correlation_pairs'])}\n")
                f.write(f"异常值较多的特征数: {len(feature_analysis['features_with_many_outliers'])}\n")
                
                if feature_analysis['zero_variance_features']:
                    f.write(f"\n零方差特征索引: {feature_analysis['zero_variance_features'][:20]}\n")
                if feature_analysis['high_correlation_pairs']:
                    f.write(f"\n高相关特征对 (前10):\n")
                    for i, (f1, f2, corr) in enumerate(feature_analysis['high_correlation_pairs'][:10]):
                        f.write(f"  特征{f1} - 特征{f2}: {corr:.4f}\n")
                
                # 标准化效果分析
                if 'normalization_effect' in feature_analysis:
                    norm_effect = feature_analysis['normalization_effect']
                    f.write(f"\n标准化效果对比:\n")
                    f.write(f"  标准化前特征标准差范围: [{norm_effect['original_feature_std_range'][0]:.3f}, {norm_effect['original_feature_std_range'][1]:.3f}]\n")
                    f.write(f"  标准化后特征标准差范围: [{norm_effect['normalized_feature_std_range'][0]:.6f}, {norm_effect['normalized_feature_std_range'][1]:.6f}]\n")
                    f.write(f"  特征均值标准化效果: 原始变异 → 接近零变异\n")
                    f.write(f"  特征量纲统一: 所有特征现在具有相似的数值范围\n")
            
            # 训练建议
            f.write("\n7. 训练策略建议\n")
            f.write("-" * 30 + "\n")
            
            # 根据分析结果给出具体建议
            imbalance_ratio = class_dist['imbalance_ratio']
            missing_count = class_dist['missing_classes_count']
            
            f.write("数据预处理建议:\n")
            if self.normalization_applied:
                f.write("  ✅ 数据已进行Z-score标准化，特征量纲统一\n")
                f.write("  📊 建议在测试时使用相同的标准化参数\n")
                f.write("  🔧 可考虑进一步的特征选择或降维\n")
            
            if quality['has_nan'] or quality['has_inf']:
                f.write("  ⚠️ 处理NaN和无穷值\n")
            if quality['zero_samples'] > 0:
                f.write("  ⚠️ 移除或处理全零样本\n")
            
            f.write("\n类别权重建议:\n")
            if missing_count > 0:
                f.write(f"  ⚠️ {missing_count}个类别无样本，考虑:\n")
                f.write("     - 数据增强\n")
                f.write("     - 合并相似类别\n")
                f.write("     - 使用高权重策略\n")
            
            if imbalance_ratio > 10:
                f.write("  🎯 极度不平衡，强烈建议:\n")
                f.write("     - 使用反频率权重或有效样本数权重\n")
                f.write("     - 考虑focal loss\n")
                f.write("     - 数据重采样(SMOTE/ADASYN)\n")
            elif imbalance_ratio > 5:
                f.write("  📊 严重不平衡，建议:\n")
                f.write("     - 使用平方根平衡权重\n")
                f.write("     - 分层采样\n")
            elif imbalance_ratio > 2:
                f.write("  📈 中等不平衡，建议:\n")
                f.write("     - 使用标准平衡权重\n")
            else:
                f.write("  ✅ 数据相对平衡\n")
            
            f.write("\n模型训练建议:\n")
            f.write("  📚 批次采样策略:\n")
            if imbalance_ratio > 5:
                f.write("     - 使用BalancedBatchSampler\n")
                f.write("     - 每个批次确保类别平衡\n")
            
            f.write("  📊 验证策略:\n")
            f.write("     - 使用分层K折交叉验证\n")
            f.write("     - 监控每个类别的F1分数\n")
            f.write("     - 关注少数类别的召回率\n")
            
            f.write("  🎯 评估指标:\n")
            f.write("     - 主要指标: F1-macro, 平衡准确率\n")
            f.write("     - 辅助指标: F1-weighted, Cohen's Kappa\n")
            f.write("     - 避免单独使用准确率\n")
            
            if 'prob_idx_analysis' in class_dist:
                f.write("  🔄 prob_idx利用:\n")
                f.write("     - 考虑按prob_idx分组训练\n")
                f.write("     - 或者作为额外特征输入\n")
                f.write("     - 分析不同组的性能差异\n")
            
            if self.normalization_applied:
                f.write("  🔧 标准化相关建议:\n")
                f.write("     - 确保测试数据使用相同的标准化参数\n")
                f.write("     - 标准化有助于梯度稳定和收敛速度\n")
                f.write("     - 考虑在标准化基础上应用正则化技术\n")
        
        print(f"✅ 分析报告已保存到: {save_path}")
        return self
    
    def visualize_data_distribution(self, save_dir="./analysis_plots/"):
        """生成数据分布可视化图表"""
        import os
        os.makedirs(save_dir, exist_ok=True)
        
        print(f"\n📊 生成可视化图表...")
        
        # 设置字体（英文）
        plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
        plt.rcParams['axes.unicode_minus'] = False
        
        # 1. 类别分布图
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        
        class_counts = self.analysis_results['class_distribution']['class_counts']
        classes = list(class_counts.keys())
        counts = list(class_counts.values())
        
        # 类别样本数分布
        axes[0, 0].bar(range(len(classes)), counts, alpha=0.7, color='skyblue')
        axes[0, 0].set_title('Sample Count Distribution by Class', fontsize=14, fontweight='bold')
        axes[0, 0].set_xlabel('Class ID')
        axes[0, 0].set_ylabel('Sample Count')
        axes[0, 0].tick_params(axis='x', rotation=45)
        
        # 样本数分布直方图
        axes[0, 1].hist(counts, bins=30, alpha=0.7, color='lightgreen', edgecolor='black')
        axes[0, 1].set_title('Distribution of Sample Counts', fontsize=14, fontweight='bold')
        axes[0, 1].set_xlabel('Sample Count')
        axes[0, 1].set_ylabel('Number of Classes')
        axes[0, 1].axvline(np.mean(counts), color='red', linestyle='--', label=f'Mean: {np.mean(counts):.1f}')
        axes[0, 1].axvline(np.median(counts), color='orange', linestyle='--', label=f'Median: {np.median(counts):.1f}')
        axes[0, 1].legend()
        
        # prob_idx分布
        prob_idx_analysis = self.analysis_results['class_distribution']['prob_idx_analysis']
        prob_values = list(prob_idx_analysis.keys())
        prob_counts = [prob_idx_analysis[p]['sample_count'] for p in prob_values]
        
        axes[1, 0].pie(prob_counts, labels=[f'prob_idx={p}' for p in prob_values], 
                      autopct='%1.1f%%', startangle=90)
        axes[1, 0].set_title('Sample Distribution by prob_idx', fontsize=14, fontweight='bold')
        
        # 累积分布
        sorted_counts = sorted(counts)
        cumulative_pct = np.arange(1, len(sorted_counts) + 1) / len(sorted_counts) * 100
        
        axes[1, 1].plot(sorted_counts, cumulative_pct, marker='o', linewidth=2, markersize=4)
        axes[1, 1].set_title('Cumulative Distribution of Sample Counts', fontsize=14, fontweight='bold')
        axes[1, 1].set_xlabel('Sample Count')
        axes[1, 1].set_ylabel('Cumulative Percentage')
        axes[1, 1].grid(True, alpha=0.3)
        
        # 添加重要统计信息
        norm_status = "✅ Normalized" if self.normalization_applied else "❌ Not Normalized"
        stats_text = f"""Statistics:
Total Classes: {len(classes)}
Sample Range: {min(counts)} - {max(counts)}
Imbalance Ratio: {max(counts)/min(counts):.2f}
Missing Classes: {self.analysis_results['class_distribution']['missing_classes_count']}
Data Status: {norm_status}"""
        
        fig.text(0.02, 0.02, stats_text, fontsize=10, 
                bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
        
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, 'class_distribution.png'), dpi=300, bbox_inches='tight')
        plt.show()
        
        # 2. 权重策略比较图
        if 'weight_strategies' in self.analysis_results:
            fig, axes = plt.subplots(2, 3, figsize=(18, 12))
            axes = axes.flatten()
            
            weights = self.analysis_results['weight_strategies']
            strategy_names = list(weights.keys())
            
            for i, (strategy_name, strategy_weights) in enumerate(weights.items()):
                if i >= 6:  # 最多显示6个策略
                    break
                    
                if strategy_weights:
                    weight_classes = list(strategy_weights.keys())
                    weight_values = list(strategy_weights.values())
                    
                    # 只显示有样本的类别的权重
                    present_classes = list(class_counts.keys())
                    present_weights = [strategy_weights.get(c, 0) for c in present_classes]
                    
                    axes[i].bar(range(len(present_classes)), present_weights, alpha=0.7)
                    axes[i].set_title(f'{strategy_name} Weight Strategy', fontsize=12, fontweight='bold')
                    axes[i].set_xlabel('Class ID')
                    axes[i].set_ylabel('Weight Value')
                    axes[i].tick_params(axis='x', rotation=45)
                    
                    # 标注统计信息
                    axes[i].text(0.02, 0.98, f'Range: [{min(present_weights):.3f}, {max(present_weights):.3f}]', 
                               transform=axes[i].transAxes, verticalalignment='top',
                               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
            
            # 隐藏多余的子图
            for i in range(len(strategy_names), 6):
                axes[i].set_visible(False)
            
            plt.tight_layout()
            plt.savefig(os.path.join(save_dir, 'weight_strategies.png'), dpi=300, bbox_inches='tight')
            plt.show()
        
        # 3. 特征分析图
        if 'feature_analysis' in self.analysis_results:
            feature_analysis = self.analysis_results['feature_analysis']
            
            fig, axes = plt.subplots(2, 2, figsize=(16, 12))
            
            # 特征均值分布
            axes[0, 0].hist(feature_analysis['means'], bins=50, alpha=0.7, color='lightcoral')
            axes[0, 0].set_title('Distribution of Feature Means (After Normalization)', fontsize=14, fontweight='bold')
            axes[0, 0].set_xlabel('Mean Value')
            axes[0, 0].set_ylabel('Number of Features')
            if self.normalization_applied:
                axes[0, 0].text(0.02, 0.98, 'Z-score normalized', transform=axes[0, 0].transAxes, 
                               verticalalignment='top', bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))
            
            # 特征标准差分布
            axes[0, 1].hist(feature_analysis['stds'], bins=50, alpha=0.7, color='lightblue')
            axes[0, 1].set_title('Distribution of Feature Standard Deviations', fontsize=14, fontweight='bold')
            axes[0, 1].set_xlabel('Standard Deviation')
            axes[0, 1].set_ylabel('Number of Features')
            axes[0, 1].axvline(np.mean(feature_analysis['stds']), color='red', linestyle='--', 
                              label=f'Mean: {np.mean(feature_analysis["stds"]):.4f}')
            axes[0, 1].legend()
            
            # 异常值分布
            outliers = feature_analysis['outliers_per_feature']
            axes[1, 0].hist(outliers, bins=30, alpha=0.7, color='lightyellow', edgecolor='black')
            axes[1, 0].set_title('Distribution of Outliers per Feature', fontsize=14, fontweight='bold')
            axes[1, 0].set_xlabel('Number of Outliers')
            axes[1, 0].set_ylabel('Number of Features')
            
            # 特征重要性代理图（使用方差）
            feature_importance_proxy = feature_analysis['stds']
            top_features = np.argsort(feature_importance_proxy)[-20:]  # 方差最大的20个特征
            
            axes[1, 1].bar(range(20), feature_importance_proxy[top_features], alpha=0.7, color='lightgreen')
            axes[1, 1].set_title('High Variance Features (Top 20)', fontsize=14, fontweight='bold')
            axes[1, 1].set_xlabel('Feature Rank')
            axes[1, 1].set_ylabel('Standard Deviation')
            axes[1, 1].tick_params(axis='x', rotation=45)
            
            plt.tight_layout()
            plt.savefig(os.path.join(save_dir, 'feature_analysis.png'), dpi=300, bbox_inches='tight')
            plt.show()
        
        # 4. prob_idx详细分析图
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        prob_idx_analysis = self.analysis_results['class_distribution']['prob_idx_analysis']
        
        # 每个prob_idx的类别数和样本数
        prob_values = list(prob_idx_analysis.keys())
        prob_samples = [prob_idx_analysis[p]['sample_count'] for p in prob_values]
        prob_classes = [prob_idx_analysis[p]['unique_classes'] for p in prob_values]
        
        x_pos = np.arange(len(prob_values))
        width = 0.35
        
        bars1 = axes[0].bar(x_pos - width/2, prob_samples, width, label='Sample Count', alpha=0.7, color='skyblue')
        bars2 = axes[0].bar(x_pos + width/2, prob_classes, width, label='Class Count', alpha=0.7, color='lightcoral')
        
        axes[0].set_title('Sample Count and Class Count by prob_idx', fontsize=14, fontweight='bold')
        axes[0].set_xlabel('prob_idx')
        axes[0].set_ylabel('Count')
        axes[0].set_xticks(x_pos)
        axes[0].set_xticklabels(prob_values)
        axes[0].legend()
        
        # 为柱状图添加数值标签
        for bar in bars1:
            height = bar.get_height()
            axes[0].text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(height)}', ha='center', va='bottom', fontsize=10)
        
        for bar in bars2:
            height = bar.get_height()
            axes[0].text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(height)}', ha='center', va='bottom', fontsize=10)
        
        # 各prob_idx组内的类别分布差异
        prob_diversities = []
        for prob_val in prob_values:
            class_dist = prob_idx_analysis[prob_val]['class_distribution']
            if class_dist:
                counts = list(class_dist.values())
                diversity = np.std(counts) / np.mean(counts) if np.mean(counts) > 0 else 0  # 变异系数
                prob_diversities.append(diversity)
            else:
                prob_diversities.append(0)
        
        axes[1].bar(prob_values, prob_diversities, alpha=0.7, color='lightgreen')
        axes[1].set_title('Class Imbalance within Each prob_idx Group', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('prob_idx')
        axes[1].set_ylabel('Coefficient of Variation (std/mean)')
        
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, 'prob_idx_analysis.png'), dpi=300, bbox_inches='tight')
        plt.show()
        
        # 新增：5. 标准化效果对比图（如果有原始数据）
        if self.normalization_applied and self.original_data is not None:
            self._plot_normalization_comparison(save_dir)
        
        print(f"✅ 所有可视化图表已保存到: {save_dir}")
        return self
    
    def _plot_normalization_comparison(self, save_dir):
        """生成标准化前后对比的可视化"""
        print("   📊 生成标准化效果对比图...")
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # 选择几个代表性特征进行对比
        n_features_to_show = min(6, self.data.shape[1])
        feature_indices = np.linspace(0, self.data.shape[1]-1, n_features_to_show, dtype=int)
        
        for i, feature_idx in enumerate(feature_indices):
            row = i // 3
            col = i % 3
            
            # 原始数据分布
            axes[row, col].hist(self.original_data[:, feature_idx], bins=50, alpha=0.6, 
                               color='red', label='Original', density=True)
            
            # 标准化后数据分布
            axes[row, col].hist(self.data[:, feature_idx], bins=50, alpha=0.6, 
                               color='blue', label='Normalized', density=True)
            
            axes[row, col].set_title(f'Feature {feature_idx}: Before vs After Normalization', 
                                   fontsize=12, fontweight='bold')
            axes[row, col].set_xlabel('Value')
            axes[row, col].set_ylabel('Density')
            axes[row, col].legend()
            axes[row, col].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, 'normalization_comparison.png'), dpi=300, bbox_inches='tight')
        plt.show()
        
        # 整体特征统计对比图
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        # 特征均值对比
        original_means = np.mean(self.original_data, axis=0)
        normalized_means = np.mean(self.data, axis=0)
        
        axes[0].scatter(original_means, normalized_means, alpha=0.6, s=10)
        axes[0].set_title('Feature Means: Original vs Normalized', fontsize=14, fontweight='bold')
        axes[0].set_xlabel('Original Mean')
        axes[0].set_ylabel('Normalized Mean')
        axes[0].grid(True, alpha=0.3)
        # 添加参考线
        axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.7, label='Expected mean ≈ 0')
        axes[0].legend()
        
        # 特征标准差对比
        original_stds = np.std(self.original_data, axis=0)
        normalized_stds = np.std(self.data, axis=0)
        
        axes[1].scatter(original_stds, normalized_stds, alpha=0.6, s=10)
        axes[1].set_title('Feature Std: Original vs Normalized', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Original Std')
        axes[1].set_ylabel('Normalized Std')
        axes[1].grid(True, alpha=0.3)
        # 添加参考线
        axes[1].axhline(y=1, color='red', linestyle='--', alpha=0.7, label='Expected std ≈ 1')
        axes[1].legend()
        
        # 特征范围对比
        original_ranges = np.max(self.original_data, axis=0) - np.min(self.original_data, axis=0)
        normalized_ranges = np.max(self.data, axis=0) - np.min(self.data, axis=0)
        
        axes[2].scatter(original_ranges, normalized_ranges, alpha=0.6, s=10)
        axes[2].set_title('Feature Ranges: Original vs Normalized', fontsize=14, fontweight='bold')
        axes[2].set_xlabel('Original Range')
        axes[2].set_ylabel('Normalized Range')
        axes[2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, 'normalization_statistics_comparison.png'), dpi=300, bbox_inches='tight')
        plt.show()
    
    def get_recommended_weights(self, strategy='auto'):
        """获取推荐的类别权重"""
        if 'weight_strategies' not in self.analysis_results:
            print("⚠️ 请先运行calculate_class_weights()方法")
            return None
        
        weights = self.analysis_results['weight_strategies']
        
        # 优先使用外部权重文件
        external_strategy = None
        for strategy_name in weights.keys():
            if strategy_name.endswith('_external'):
                external_strategy = strategy_name
                break
        
        if external_strategy:
            recommended_weights = weights[external_strategy]
            print(f"🎯 使用外部权重: {external_strategy}")
            print(f"   来源文件: {self.weight_file_path}")
            if self.weight_metadata:
                print(f"   权重方法: {self.weight_metadata.get('method', 'unknown')}")
                print(f"   Beta参数: {self.weight_metadata.get('beta', 'unknown')}")
            strategy_used = external_strategy
        else:
            # 否则使用分析计算的权重
            imbalance_ratio = self.analysis_results['class_distribution']['imbalance_ratio']
            
            if strategy == 'auto':
                # 自动选择最适合的策略
                if imbalance_ratio <= 2:
                    strategy = 'balanced'
                elif imbalance_ratio <= 5:
                    strategy = 'sqrt_balanced'
                elif imbalance_ratio <= 10:
                    strategy = 'effective_number'
                else:
                    strategy = 'inverse_frequency'
            
            recommended_weights = weights.get(strategy, {})
            strategy_used = strategy
            
            if recommended_weights:
                print(f"🎯 推荐使用: {strategy} 权重策略")
                print(f"   不平衡比例: {imbalance_ratio:.2f}")
            
        if recommended_weights:
            print(f"   权重范围: [{min(recommended_weights.values()):.4f}, {max(recommended_weights.values()):.4f}]")
            if self.normalization_applied:
                print(f"   📊 基于标准化后的数据分析")
            
            # 转换为适合PyTorch使用的格式
            num_classes = self.analysis_results['basic_stats']['num_classes']
            weight_tensor = []
            for i in range(num_classes):
                weight_tensor.append(recommended_weights.get(i, 1.0))
            
            return {
                'strategy': strategy_used,
                'weights_dict': recommended_weights,
                'weights_tensor': weight_tensor,
                'imbalance_ratio': self.analysis_results['class_distribution']['imbalance_ratio'],
                'normalized_data': self.normalization_applied,
                'external_weights': external_strategy is not None,
                'weight_file_path': self.weight_file_path if external_strategy else None
            }
        else:
            print(f"❌ 策略 {strategy_used} 不可用")
            return None
    
    def export_weights_for_pytorch(self, strategy='auto', save_path='class_weights.py'):
        """导出适合PyTorch使用的权重代码"""
        weight_info = self.get_recommended_weights(strategy)
        
        if weight_info is None:
            return
        
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write('"""\n')
            f.write('自动生成的类别权重配置\n')
            f.write(f'分析时间: {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
            f.write(f'推荐策略: {weight_info["strategy"]}\n')
            f.write(f'不平衡比例: {weight_info["imbalance_ratio"]:.2f}\n')
            f.write(f'数据预处理: {"Z-score标准化" if weight_info["normalized_data"] else "未标准化"}\n')
            if weight_info["external_weights"]:
                f.write(f'权重来源: 外部文件 ({weight_info["weight_file_path"]})\n')
            else:
                f.write('权重来源: 实时计算\n')
            f.write('"""\n\n')
            
            f.write('import torch\n')
            f.write('import numpy as np\n')
            if weight_info["external_weights"]:
                f.write('import os\n')
            f.write('\n')
            
            # 权重张量
            f.write('# 类别权重张量 (适用于CrossEntropyLoss)\n')
            f.write(f'CLASS_WEIGHTS_TENSOR = torch.FloatTensor({weight_info["weights_tensor"]})\n\n')
            
            # 权重字典
            f.write('# 类别权重字典\n')
            f.write(f'CLASS_WEIGHTS_DICT = {weight_info["weights_dict"]}\n\n')
            
            # 配置信息
            f.write('# 配置信息\n')
            f.write('WEIGHT_CONFIG = {\n')
            f.write(f'    "strategy": "{weight_info["strategy"]}",\n')
            f.write(f'    "imbalance_ratio": {weight_info["imbalance_ratio"]:.4f},\n')
            f.write(f'    "num_classes": {len(weight_info["weights_tensor"])},\n')
            f.write(f'    "min_weight": {min(weight_info["weights_tensor"]):.6f},\n')
            f.write(f'    "max_weight": {max(weight_info["weights_tensor"]):.6f},\n')
            f.write(f'    "mean_weight": {np.mean(weight_info["weights_tensor"]):.6f},\n')
            f.write(f'    "data_normalized": {weight_info["normalized_data"]},\n')
            f.write(f'    "external_weights": {weight_info["external_weights"]}\n')
            if weight_info["external_weights"]:
                f.write(f'    "weight_file_path": "{weight_info["weight_file_path"]}",\n')
            f.write('}\n\n')
            
            # 使用示例
            f.write('# 使用示例:\n')
            f.write('# criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS_TENSOR.to(device))\n')
            f.write('# \n')
            f.write('# 或者在自定义损失函数中使用:\n')
            f.write('# def weighted_cross_entropy(outputs, targets, weights=CLASS_WEIGHTS_TENSOR):\n')
            f.write('#     return F.cross_entropy(outputs, targets, weight=weights.to(outputs.device))\n')
            if weight_info["normalized_data"]:
                f.write('# \n')
                f.write('# 注意: 这些权重是基于Z-score标准化后的数据计算的\n')
                f.write('# 请确保在训练和测试时都使用相同的标准化参数\n')
            if weight_info["external_weights"]:
                f.write('# \n')
                f.write('# 权重来源: 外部权重文件\n')
                f.write(f'# 原始文件: {weight_info["weight_file_path"]}\n')
                f.write('# 如需更新权重，请修改原始权重文件并重新运行分析\n')
        
        print(f"✅ 权重配置已导出到: {save_path}")
        return weight_info

# 使用示例和主函数
def analyze_mat_file(mat_file_path, weight_config=None, output_dir="./mat_analysis_output/"):
    """
    完整的MAT文件分析流程
    
    Args:
        mat_file_path: MAT文件路径
        weight_config: 权重配置字典或配置文件路径
        output_dir: 输出目录
    """
    import os
    import json
    os.makedirs(output_dir, exist_ok=True)
    
    print("🚀 开始MAT文件深度分析...")
    print("=" * 60)
    
    # 处理权重配置
    weight_file_path = None
    if weight_config:
        if isinstance(weight_config, str):
            # 可能是配置文件路径或直接的权重文件路径
            if weight_config.endswith(('.json', '.yaml', '.yml')):
                # 配置文件
                pass
            else:
                # 直接的权重文件路径
                weight_file_path = weight_config
                weight_config = None
        elif isinstance(weight_config, dict):
            # 权重配置字典
            if 'effective_number' in weight_config:
                weight_file_path = weight_config['effective_number'].get('file_path')
    
    # 创建分析器
    analyzer = ComprehensiveMatAnalysis(mat_file_path, weight_file_path, weight_config)
    
    # 执行完整分析流程
    try:
        # 1. 数据加载 (自动包含标准化和权重配置处理)
        analyzer.load_data()
        
        # 2. 基础统计分析
        analyzer.analyze_basic_statistics()
        
        # 3. 类别分布分析
        analyzer.analyze_class_distribution()
        
        # 4. 权重计算
        analyzer.calculate_class_weights()
        
        # 5. 特征分析
        analyzer.analyze_feature_distribution()
        
        # 6. 生成报告
        report_path = os.path.join(output_dir, "comprehensive_analysis_report.txt")
        analyzer.generate_comprehensive_report(report_path)
        
        # 7. 生成可视化
        viz_dir = os.path.join(output_dir, "visualizations")
        analyzer.visualize_data_distribution(viz_dir)
        
        # 8. 导出权重配置
        weights_path = os.path.join(output_dir, "recommended_class_weights.py")
        weight_info = analyzer.export_weights_for_pytorch(save_path=weights_path)
        
        # 9. 如果没有外部权重文件，可选择性导出当前最佳权重到文件
        if not analyzer.external_weights and weight_info:
            weight_file_path = os.path.join(output_dir, f"weights_{weight_info['strategy']}.json")
            analyzer.export_weights_to_file(weight_file_path, weight_info['strategy'])
        
        print("\n" + "=" * 60)
        print("🎉 分析完成!")
        print("=" * 60)
        print(f"📁 输出目录: {output_dir}")
        print(f"📊 分析报告: {report_path}")
        print(f"📈 可视化图表: {viz_dir}")
        print(f"⚖️ 权重配置: {weights_path}")
        
        if analyzer.normalization_applied:
            print(f"\n🔧 数据预处理: Z-score标准化已应用")
            print(f"📊 特征维度: {analyzer.data.shape[1]} (全部标准化)")
        
        if analyzer.external_weights:
            print(f"\n📂 外部权重: 已加载 {analyzer.weight_file_path}")
            if analyzer.weight_metadata:
                print(f"📊 权重方法: {analyzer.weight_metadata.get('method', 'unknown')}")
        
        if weight_info:
            print(f"\n🎯 推荐权重策略: {weight_info['strategy']}")
            print(f"📊 不平衡比例: {weight_info['imbalance_ratio']:.2f}")
            print(f"⚖️ 权重范围: [{min(weight_info['weights_tensor']):.4f}, {max(weight_info['weights_tensor']):.4f}]")
            if weight_info['external_weights']:
                print(f"📄 权重来源: 外部文件")
            else:
                print(f"🔄 权重来源: 实时计算")
        
        return analyzer
        
    except Exception as e:
        print(f"❌ 分析过程中出现错误: {str(e)}")
        import traceback
        traceback.print_exc()
        return None



# 权重文件生成工具函数
def generate_effective_number_weights(class_counts, output_path, beta=0.999, metadata=None):
    """
    生成Effective Number权重文件的独立工具
    
    Args:
        class_counts: list or dict, 每个类别的样本数量
        output_path: str, 输出文件路径
        beta: float, 超参数
        metadata: dict, 额外的元数据
    """
    import json
    import numpy as np
    import pandas as pd
    import os
    
    if isinstance(class_counts, dict):
        # 如果是字典，转换为列表
        max_class_id = max(class_counts.keys())
        counts_list = []
        for i in range(max_class_id + 1):
            counts_list.append(class_counts.get(i, 0))
        class_counts = counts_list
    
    num_classes = len(class_counts)
    
    # 计算Effective Number权重
    effective_weights = {}
    weight_tensor = []
    
    for class_id, count in enumerate(class_counts):
        if count > 0:
            effective_num = (1 - beta**count) / (1 - beta)
            weight = 1 / effective_num
        else:
            # 为零样本类别分配高权重
            weight = 1 / (1 - beta)
        
        effective_weights[class_id] = weight
        weight_tensor.append(weight)
    
    # 归一化权重（使总和等于类别数）
    weight_array = np.array(weight_tensor)
    normalized_weights = weight_array / np.mean(weight_array)
    
    # 更新权重
    for i, weight in enumerate(normalized_weights):
        effective_weights[i] = float(weight)
        weight_tensor[i] = float(weight)
    
    # 准备元数据
    default_metadata = {
        'method': 'effective_number',
        'beta': beta,
        'num_classes': num_classes,
        'creation_time': pd.Timestamp.now().isoformat(),
        'imbalance_ratio': float(max(class_counts) / max(1, min([c for c in class_counts if c > 0]))) if any(c > 0 for c in class_counts) else 1.0
    }
    
    if metadata:
        default_metadata.update(metadata)
    
    # 构建输出数据
    output_data = {
        'metadata': default_metadata,
        'class_weights': {str(k): v for k, v in effective_weights.items()},
        'weight_tensor': weight_tensor
    }
    
    # 创建输出目录
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # 根据文件扩展名保存
    file_ext = os.path.splitext(output_path)[1].lower()
    
    if file_ext == '.json':
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(output_data, f, indent=2, ensure_ascii=False)
    elif file_ext == '.npy':
        np.save(output_path, np.array(weight_tensor))
    elif file_ext == '.npz':
        np.savez(output_path, weights=np.array(weight_tensor), metadata=default_metadata)
    elif file_ext == '.csv':
        df = pd.DataFrame({
            'class_id': list(range(num_classes)),
            'weight': weight_tensor,
            'sample_count': class_counts
        })
        df.to_csv(output_path, index=False)
    else:
        # 默认JSON格式
        with open(output_path + '.json', 'w', encoding='utf-8') as f:
            json.dump(output_data, f, indent=2, ensure_ascii=False)
        output_path += '.json'
    
    print(f"✅ Effective Number权重已生成: {output_path}")
    print(f"   Beta参数: {beta}")
    print(f"   类别数量: {num_classes}")
    print(f"   权重范围: [{min(weight_tensor):.4f}, {max(weight_tensor):.4f}]")
    
    return output_data

# 运行分析
if __name__ == "__main__":
    # 替换为你的MAT文件路径
    mat_file_path = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat"
    
    # 示例1: 基础分析（无外部权重）
    print("=" * 60)
    print("示例1: 基础分析")
    print("=" * 60)
    analyzer = analyze_mat_file(mat_file_path)
    
    # 示例2: 使用外部权重文件
    # print("\n" + "=" * 60)
    # print("示例2: 使用外部权重文件")
    # print("=" * 60)
    # 
    # # 首先生成一个示例权重文件（实际使用时可能已存在）
    # if analyzer and 'class_distribution' in analyzer.analysis_results:
    #     class_counts = analyzer.analysis_results['class_distribution']['class_counts']
    #     weights_file = "./weights/effective_number_beta0.999_TRAIN38.json"
    #     generate_effective_number_weights(class_counts, weights_file, beta=0.999)
    #     
    #     # 使用外部权重文件重新分析
    #     analyzer_with_weights = analyze_mat_file(
    #         mat_file_path, 
    #         weight_config=weights_file,
    #         output_dir="./mat_analysis_with_external_weights/"
    #     )
    
    if analyzer:
        print("\n🔍 快速访问分析结果:")
        print("analyzer.analysis_results['basic_stats']  # 基础统计")
        print("analyzer.analysis_results['class_distribution']  # 类别分布")
        print("analyzer.analysis_results['weight_strategies']  # 权重策略")
        print("analyzer.get_recommended_weights()  # 获取推荐权重")
        print("analyzer.load_external_weights('path/to/weights.json')  # 加载外部权重")
        print("analyzer.export_weights_to_file('output.json', 'effective_number')  # 导出权重")
        print("\n📊 数据状态:")
        print(f"  - 原始数据形状: {analyzer.original_data.shape if analyzer.original_data is not None else 'N/A'}")
        print(f"  - 当前数据形状: {analyzer.data.shape}")
        print(f"  - 标准化状态: {'✅ 已标准化' if analyzer.normalization_applied else '❌ 未标准化'}")

In [ ]:
class EffectiveNumberWeights:
    """Effective Number of Samples 权重计算工具类"""
    
    @staticmethod
    def compute_weights(class_counts, beta=0.999, return_type='numpy', normalize=True):
        """
        计算 Effective Number 权重
        
        Args:
            class_counts: list/array/dict, 每个类别的样本数量
            beta: float, 超参数 (0, 1)
            return_type: str, 返回类型 ('numpy', 'torch', 'dict')
            normalize: bool, 是否归一化权重
            
        Returns:
            根据return_type返回相应格式的权重
        """
        import numpy as np
        
        # 处理输入格式
        if isinstance(class_counts, dict):
            max_class = max(class_counts.keys())
            counts_array = np.zeros(max_class + 1)
            for k, v in class_counts.items():
                counts_array[k] = v
        else:
            counts_array = np.array(class_counts)
        
        num_classes = len(counts_array)
        weights = np.zeros(num_classes)
        
        # 计算每个类别的有效样本数和权重
        for i, count in enumerate(counts_array):
            if count > 0:
                # En = (1 - β^n) / (1 - β)
                effective_num = (1 - beta**count) / (1 - beta)
                weights[i] = 1 / effective_num
            else:
                # 零样本类别分配高权重
                weights[i] = 1 / (1 - beta)
        
        # 归一化权重（总和等于类别数）
        if normalize:
            weights = weights / np.mean(weights)
        
        # 返回指定格式
        if return_type == 'numpy':
            return weights
        elif return_type == 'torch':
            try:
                import torch
                return torch.FloatTensor(weights)
            except ImportError:
                raise ImportError("PyTorch未安装，无法返回torch格式")
        elif return_type == 'dict':
            return {i: float(w) for i, w in enumerate(weights)}
        else:
            raise ValueError(f"不支持的返回类型: {return_type}")
    
    @staticmethod
    def recommend_beta(class_counts, num_classes=None, dataset_type='auto'):
        """
        根据数据特性推荐beta值
        
        Args:
            class_counts: 类别样本数
            num_classes: 总类别数
            dataset_type: 数据集类型 ('fine', 'coarse', 'auto')
            
        Returns:
            推荐的beta值
        """
        if isinstance(class_counts, dict):
            counts_list = list(class_counts.values())
            if num_classes is None:
                num_classes = len(class_counts)
        else:
            counts_list = class_counts
            if num_classes is None:
                num_classes = len(counts_list)
        
        # 计算不平衡比例
        positive_counts = [c for c in counts_list if c > 0]
        if not positive_counts:
            return 0.999  # 默认值
        
        imbalance_ratio = max(positive_counts) / min(positive_counts)
        
        # 自动判断数据集类型
        if dataset_type == 'auto':
            if num_classes > 50:
                dataset_type = 'fine'
            else:
                dataset_type = 'coarse'
        
        # 根据论文建议选择beta
        if dataset_type == 'fine':
            if imbalance_ratio < 50:
                return 0.99
            elif imbalance_ratio < 100:
                return 0.999
            else:
                return 0.9999
        else:  # coarse
            if imbalance_ratio < 100:
                return 0.999
            else:
                return 0.9999
    
    @staticmethod
    def compare_beta_values(class_counts, beta_values=[0.9, 0.99, 0.999, 0.9999]):
        """
        比较不同beta值的权重分布
        
        Args:
            class_counts: 类别样本数
            beta_values: 要比较的beta值列表
            
        Returns:
            字典，包含每个beta值对应的权重
        """
        results = {}
        
        for beta in beta_values:
            weights = EffectiveNumberWeights.compute_weights(
                class_counts, beta=beta, return_type='dict'
            )
            weight_values = list(weights.values())
            
            results[beta] = {
                'weights': weights,
                'min_weight': min(weight_values),
                'max_weight': max(weight_values),
                'weight_ratio': max(weight_values) / min(weight_values),
                'mean_weight': np.mean(weight_values),
                'std_weight': np.std(weight_values)
            }
        
        return results

# 使用示例函数
def create_pytorch_weighted_loss(class_counts, beta='auto', loss_type='crossentropy'):
    """
    创建带有Effective Number权重的PyTorch损失函数
    
    Args:
        class_counts: 类别样本数
        beta: 超参数，'auto'表示自动选择
        loss_type: 损失函数类型 ('crossentropy', 'focal')
        
    Returns:
        配置好的损失函数
    """
    try:
        import torch
        import torch.nn as nn
    except ImportError:
        raise ImportError("需要安装PyTorch")
    
    # 推荐beta值
    if beta == 'auto':
        beta = EffectiveNumberWeights.recommend_beta(class_counts)
        print(f"🎯 自动选择beta值: {beta}")
    
    # 计算权重
    weights = EffectiveNumberWeights.compute_weights(
        class_counts, beta=beta, return_type='torch'
    )
    
    print(f"📊 权重范围: [{weights.min():.4f}, {weights.max():.4f}]")
    
    # 创建损失函数
    if loss_type == 'crossentropy':
        criterion = nn.CrossEntropyLoss(weight=weights)
    elif loss_type == 'focal':
        # 这里需要自定义Focal Loss实现
        class FocalLoss(nn.Module):
            def __init__(self, weight=None, gamma=2.0):
                super().__init__()
                self.weight = weight
                self.gamma = gamma
                
            def forward(self, inputs, targets):
                ce_loss = nn.functional.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
                pt = torch.exp(-ce_loss)
                focal_loss = (1 - pt) ** self.gamma * ce_loss
                return focal_loss.mean()
        
        criterion = FocalLoss(weight=weights)
    else:
        raise ValueError(f"不支持的损失函数类型: {loss_type}")
    
    return criterion, weights, beta


# 示例：直接使用Effective Number权重
if __name__ == "__main__":
    # 示例类别分布（高度不平衡）
    class_counts = [1000, 500, 100, 50, 10, 5, 1, 0, 0, 0]
    
    print("🔍 Effective Number权重计算示例:")
    print("=" * 50)
    
    # 1. 基础使用
    weights_numpy = EffectiveNumberWeights.compute_weights(class_counts, beta=0.999)
    print(f"NumPy权重: {weights_numpy}")
    
    # 2. 自动beta选择
    recommended_beta = EffectiveNumberWeights.recommend_beta(class_counts)
    print(f"推荐beta值: {recommended_beta}")
    
    # 3. PyTorch使用
    try:
        criterion, weights, beta = create_pytorch_weighted_loss(class_counts, beta='auto')
        print(f"PyTorch损失函数已创建，使用beta={beta}")
    except ImportError:
        print("PyTorch未安装，跳过PyTorch示例")
    
    # 4. beta值比较
    beta_comparison = EffectiveNumberWeights.compare_beta_values(class_counts)
    print("\n📊 不同beta值权重对比:")
    for beta, stats in beta_comparison.items():
        print(f"  β={beta}: 权重比例={stats['weight_ratio']:.2f}, 均值={stats['mean_weight']:.3f}")